# Cat vs Dog Demo Application

This notebook loads the trained **ViT-Small head-only** model by default and provides a small demo app for classifying uploaded images as `cat` or `dog`.

Default checkpoint: `models_checkpoint/vit_small_head.pt`

## 1. Install and Import Dependencies

In [1]:
!pip -q install timm gradio

from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
import timm
import gradio as gr

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.6 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.19.0 which is incompatible.


/home/datnguyen/anaconda3/envs/general/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## 2. Configuration

The model architecture is fixed to `vit_small_patch16_224` with a binary classification head. If you upload or move the checkpoint in Colab, update `CHECKPOINT_PATH`.

In [6]:
IMG_SIZE = 224
MODEL_NAME = 'vit_small_patch16_224'
CLASS_NAMES = ['cat', 'dog']

def find_default_checkpoint():
    candidates = [
        Path('models_checkpoint/vit_small_head.pt'),
        Path('exercise_1/models_checkpoint/vit_small_head.pt'),
        Path('/content/models_checkpoint/vit_small_head.pt'),
        Path('/content/vit_small_head.pt'),
        Path('/content/models/vit_small_head.pt'),
    ]
    candidates.extend(Path.cwd().glob('**/models_checkpoint/vit_small_head.pt'))
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]

CHECKPOINT_PATH = find_default_checkpoint()
print('Default checkpoint:', CHECKPOINT_PATH)

Default checkpoint: models_checkpoint/vit_small_head.pt


## 3. Load Model

In [7]:
def load_checkpoint(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {path}')
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

def create_vit_head_model():
    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=len(CLASS_NAMES))
    return model

def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict) and 'model_state' in checkpoint:
        return checkpoint['model_state']
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        return checkpoint['state_dict']
    return checkpoint

def load_model(checkpoint_path=CHECKPOINT_PATH):
    checkpoint = load_checkpoint(checkpoint_path)
    model = create_vit_head_model()
    state_dict = extract_state_dict(checkpoint)
    model.load_state_dict(state_dict)
    model.to(DEVICE)
    model.eval()
    return model

model = load_model(CHECKPOINT_PATH)
print('Loaded model:', MODEL_NAME)
print('Loaded checkpoint:', CHECKPOINT_PATH)

Loaded model: vit_small_patch16_224
Loaded checkpoint: models_checkpoint/vit_small_head.pt


## 4. Prediction Function

In [8]:
preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def predict_image(image):
    if image is None:
        return None
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    image = image.convert('RGB')
    x = preprocess(image).unsqueeze(0).to(DEVICE)
    logits = model(x)
    probs = torch.softmax(logits, dim=1)[0].detach().cpu()
    return {CLASS_NAMES[i]: float(probs[i]) for i in range(len(CLASS_NAMES))}

def predict_image_path(image_path):
    scores = predict_image(Image.open(image_path))
    label = max(scores, key=scores.get)
    confidence = scores[label]
    print(f'Prediction: {label}')
    print(f'Confidence: {confidence:.4f}')
    return label, confidence

# Example:
# predict_image_path('/content/example.jpg')

/home/datnguyen/anaconda3/envs/general/lib/python3.11/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## 5. Demo App

Run this cell in Google Colab. Upload a cat or dog image in the interface, and the app will return class probabilities.

In [5]:
demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type='pil', label='Upload image'),
    outputs=gr.Label(num_top_classes=2, label='Prediction'),
    title='Cat vs Dog Classifier',
    description='Default model: vit_small_patch16_224 head fine-tuned checkpoint.',
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Missing file: /home/datnguyen/.cache/huggingface/gradio/frpc/frpc_linux_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64
2. Rename the downloaded file to: frpc_linux_amd64_v0.3
3. Move the file to this location: /home/datnguyen/.cache/huggingface/gradio/frpc
